# Class 07 · Part 2: Correcting a bias with machine learning

A reanalysis like ERA5 (class 05) gives the temperature of a **cell**: the mean over an area, at the mean
height of that area. A station measures at one **point**. On a mountain the two are very different (class 06).
Can a model learn to turn the ERA5 value into the value of the station? That is a **bias correction**
(also called statistical **downscaling**: from a cell to a point). In this notebook you

1. compare ERA5 with the DWD station on the **Zugspitze** (2 956 m), month by month since 1950,
2. train two corrections on 1950–1990 and test them on 1991–2023,
3. evaluate them with the **mean bias** and the **RMSE**, and
4. check whether the bias stays the same in a changing climate.

## 1. Station and reanalysis

The station from the DWD (class 02), as monthly means; ERA5 from the catalog (class 05), monthly, at zoom 8,
the finest of the catalog: about 25 km. `hp.ang2pix` gives the cell of the station (classes 05 and 06).

In [ ]:
import zipfile

import healpix as hp
import intake
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import download_dwd_daily


def read_station(number):
    # the daily data of a DWD station, with dates as index (class 02)
    zip_file = zipfile.ZipFile(download_dwd_daily(number))
    data_name = [
        name for name in zip_file.namelist() if name.startswith("produkt_klima_tag")
    ][0]
    data = pd.read_csv(
        zip_file.open(data_name), sep=";", skipinitialspace=True, na_values=[-999]
    )
    data["date"] = pd.to_datetime(data["MESS_DATUM"].astype(str), format="%Y%m%d")
    return data.set_index("date")


lon, lat = 10.9848, 47.4210  # Zugspitze, from the list of stations
station = read_station(5792)["TMK"].resample("MS").mean().loc["1950":"2023"]

cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")
era5 = cat.ERA5(zoom=8).to_dask()["2t"].sel(time=slice("1950", "2023"))
cell = hp.ang2pix(2**8, lon, lat, nest=True, lonlat=True)
era5_cell = (
    era5.isel(cell=cell).compute().to_series() - 273.15
)  # K to °C, as a pandas Series

Both in one table: the dates of the station and of ERA5 match (the first of each month). `dropna` removes
months that one of them does not have.

In [ ]:
data = pd.DataFrame({"era5": era5_cell, "station": station}).dropna()
print(len(data), "months")
data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
data["station"].loc["2000":"2010"].plot(ax=ax, color="black", label="station Zugspitze")
data["era5"].loc["2000":"2010"].plot(ax=ax, color="tab:blue", label="ERA5 cell")
ax.set_ylabel("temperature / °C")
ax.set_title("Monthly mean temperature, Zugspitze")
ax.legend()
plt.show()

print(f"mean bias ERA5 - station: {(data['era5'] - data['station']).mean():+.2f} °C")
print(f"correlation: {data.corr().iloc[0, 1]:.3f}")

ERA5 is about **10 °C too warm** on the Zugspitze, but it goes up and down with the station: high
correlation. A reanalysis follows the real weather (class 05), so month by month the two belong together. That
is what makes a correction possible.

## 2. Training and test periods

The correction is learned on 1950–1990 and tested on **1991–2023**, independent years it has not seen.

In [ ]:
train = data.index < "1991-01-01"
test = ~train
X, y = data[["era5"]], data["station"]  # feature: ERA5; target: the station

## 3. Two corrections

1. **Mean shift**: subtract the mean bias of the training years. The simplest correction.
2. **Linear regression**: station = $a$ × ERA5 + $b$. It can also change the **size** of the variations, e.g. a
   seasonal cycle that is too large in ERA5.

In [ ]:
from sklearn.linear_model import LinearRegression

shift = (y[train] - X["era5"][train]).mean()
shifted = X["era5"][test] + shift

model = LinearRegression().fit(X[train], y[train])
corrected = pd.Series(model.predict(X[test]), index=y[test].index)

print(f"mean shift: {shift:+.2f} °C")
print(f"regression: station = {model.coef_[0]:.3f} × ERA5 {model.intercept_:+.2f}")

## 4. Evaluate on the test years

Two numbers: the **mean bias** (is it too warm or too cold on average?) and the **RMSE** (how large are the
errors of single months?).

In [ ]:
from sklearn.metrics import mean_squared_error

results = {}
for name, values in [
    ("ERA5", X["era5"][test]),
    ("mean shift", shifted),
    ("regression", corrected),
]:
    results[name] = {
        "mean bias / °C": (values - y[test]).mean(),
        "RMSE / °C": np.sqrt(mean_squared_error(y[test], values)),
    }
pd.DataFrame(results).T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1]})
period = slice("2015", "2023")
y[test].loc[period].plot(ax=axes[0], color="black", label="station")
X["era5"][test].loc[period].plot(ax=axes[0], color="tab:blue", label="ERA5")
shifted.loc[period].plot(ax=axes[0], color="tab:green", label="mean shift")
corrected.loc[period].plot(ax=axes[0], color="firebrick", label="regression")
axes[0].set_ylabel("temperature / °C")
axes[0].set_title("Zugspitze, test years")
axes[0].legend(ncol=4)

axes[1].scatter(X["era5"][train], y[train], s=3, color="gray", label="training months")
line = np.linspace(X["era5"].min(), X["era5"].max(), 10)
axes[1].plot(
    line,
    model.coef_[0] * line + model.intercept_,
    color="firebrick",
    label="regression",
)
axes[1].plot(line, line + shift, color="tab:green", label="mean shift")
axes[1].set_xlabel("ERA5 / °C")
axes[1].set_ylabel("station / °C")
axes[1].legend()

fig.savefig(
    "/app/pages/07_machine_learning/bias_correction.png", dpi=100, bbox_inches="tight"
)
plt.show()

Both corrections remove most of the bias. The regression also fixes the size of the seasonal cycle: its slope
is smaller than 1, because ERA5 swings more between winter and summer than the summit does.

## 5. Does the bias stay the same?

A correction learned in the past assumes that the bias stays the same. Does it? The mean bias of each decade:

In [ ]:
bias = data["era5"] - data["station"]
decades = bias.groupby((bias.index.year // 10) * 10).mean()

fig, ax = plt.subplots(figsize=(8, 4))
decades.plot(ax=ax, kind="bar", color="tab:blue")
ax.set_ylim(decades.min() - 0.5, decades.max() + 0.5)
ax.set_xlabel("decade")
ax.set_ylabel("ERA5 − station / °C")
ax.set_title("Mean bias of ERA5 at the Zugspitze, by decade")

fig.savefig(
    "/app/pages/07_machine_learning/bias_by_decade.png", dpi=100, bbox_inches="tight"
)
plt.show()

The bias grows, from about 9 °C in the 1950s to about 10 °C in the 2020s. Maybe the cell (lower, on average)
warms faster than the summit; maybe the reanalysis itself changed, e.g. the observations it uses. Either way: a
correction learned on 1950–1990 is off in 1991–2023, and could be more off in a much warmer
future climate. **A model that worked in the past may not generalize to a changing climate**: that is why the
test years must be independent of the training years, and why corrections of future projections are uncertain.

## 6. From grids to samples

Here one cell and one station gave one feature and one target: `(samples,) = (months,)`. A whole grid works the
same way: data with dimensions `(time, cell)` becomes a table of **samples** (rows) and **features** (columns),
e.g. one row per time and one column per cell, or one model per cell. For the algorithm it makes no difference
whether the cells came from a lon-lat grid (class 03) or from HEALPix (classes 05 and 06). What matters is that
features and targets refer to the **same times and places**.

**Why not ICON?** A climate simulation makes its own weather (class 05): its January 2015 is not the real
January 2015, so month-by-month pairs of model and station mean nothing, and a regression like this one cannot be
trained. Free-running models are corrected by their **statistics** instead, e.g. the mean and spread of each month.

## 7. Exercises

1. **Hamburg** (station 1975, longitude 10.0, latitude 53.63): the same correction. How large is the bias there, and
   why so different from the Zugspitze?
2. **Is the bias the same all year?** Compute the mean bias of each calendar month on the Zugspitze
   (`bias.groupby(bias.index.month).mean()`). In which season is ERA5 most off?
3. **Another mountain station**, e.g. Hohenpeißenberg (2290, 977 m, 11.0108, 47.8009), Brocken (722, 1 135 m,
   10.6183, 51.7986) or Feldberg/Schwarzwald (1346, 1 486 m, 8.0038, 47.8748). Is the bias larger on higher stations?

In [ ]:
# 1. Hamburg

In [ ]:
# 2. the bias of each month

In [ ]:
# 3. another mountain station

## Your page

Open `pages/07_machine_learning/my_page.rst` in JupyterLab, write your name, and describe your results.
Look at it on http://localhost:8000 → *Classes → 07*.